# Trying same approach as TAsk 2

In [3]:
import os
import uuid
import time
import warnings
import numpy as np
import pandas as pd

import dimod
from dimod import BinaryQuadraticModel
from neal import SimulatedAnnealingSampler

from qclef import qa_access as qa

from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import ndcg_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")


# =========================================================
# CONFIG
# =========================================================

CFG = {

    # -----------------------------------
    # PATHS
    # -----------------------------------

    "data_dir":
        "/config/workspace/datasets/Tasks/1A",

    "submit_dir":
        "/config/workspace/submissions",

    # -----------------------------------
    # FILES
    # -----------------------------------

    "train_file":
        "MQ2007_train.csv",

    "dev_file":
        "MQ2007_dev.csv",

    # -----------------------------------
    # DATA
    # -----------------------------------

    "n_features":
        46,

    # -----------------------------------
    # QUBO
    # -----------------------------------

    "k_values":
        [8, 10, 12, 15, 18, 20],

    "alpha":
        0.6,

    "beta":
        0.4,

    "delta":
        0.3,

    "corr_thresh":
        0.10,

    # -----------------------------------
    # SA
    # -----------------------------------

    "sa_num_reads":
        2000,

    "random_state":
        42,

    # -----------------------------------
    # SUBMISSION
    # -----------------------------------

    "group_name":
        "FAST-NUCES",

    "method_id":
        "MI_RF_SA",
}


os.makedirs(
    CFG["submit_dir"],
    exist_ok=True
)


# =========================================================
# LOAD CSV
# =========================================================

def load_csv(path):

    df = pd.read_csv(path)

    y = df["relevance"].values.astype(np.int32)

    q = df["query_id"].values.astype(np.int32)

    feature_cols = [
        str(i)
        for i in range(1, 47)
    ]

    X = df[feature_cols].values.astype(np.float64)

    return X, y, q


# =========================================================
# GROUPS
# =========================================================

def compute_groups(query_ids):

    _, counts = np.unique(
        query_ids,
        return_counts=True
    )

    return counts.tolist()


# =========================================================
# LOAD DATA
# =========================================================

print("\nLoading data...")

X_train, y_train, q_train = load_csv(
    os.path.join(
        CFG["data_dir"],
        CFG["train_file"]
    )
)

X_dev, y_dev, q_dev = load_csv(
    os.path.join(
        CFG["data_dir"],
        CFG["dev_file"]
    )
)

print("Train shape:", X_train.shape)
print("Dev shape:", X_dev.shape)


# =========================================================
# NORMALIZATION
# =========================================================

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)

X_dev = scaler.transform(X_dev)


# =========================================================
# STEP 1 — MUTUAL INFORMATION
# =========================================================

print("\nComputing Mutual Information...")

mi = mutual_info_regression(
    X_train,
    y_train,
    random_state=CFG["random_state"]
)

mi_norm = (
    (mi - mi.min()) /
    (mi.max() - mi.min() + 1e-9)
)

print(
    "Top MI features:",
    np.argsort(mi_norm)[-5:][::-1] + 1
)


# =========================================================
# STEP 2 — RANDOM FOREST IMPORTANCE
# =========================================================

print("\nTraining Random Forest...")

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train,
    y_train
)

rf_gain = rf.feature_importances_

rf_norm = (
    (rf_gain - rf_gain.min()) /
    (rf_gain.max() - rf_gain.min() + 1e-9)
)

print(
    "Top RF features:",
    np.argsort(rf_norm)[-5:][::-1] + 1
)


# =========================================================
# STEP 3 — COMPOSITE SCORE
# =========================================================

print("\nBuilding composite score...")

score = (
    CFG["alpha"] * mi_norm +
    CFG["beta"] * rf_norm
)

score = (
    (score - score.min()) /
    (score.max() - score.min() + 1e-9)
)

print(
    "Top Composite features:",
    np.argsort(score)[-5:][::-1] + 1
)


# =========================================================
# STEP 4 — CORRELATION MATRIX
# =========================================================

print("\nComputing feature redundancy...")

corr = np.abs(
    np.corrcoef(X_train.T)
)

print(
    "Correlation matrix shape:",
    corr.shape
)


# =========================================================
# BUILD BQM
# =========================================================

def build_bqm(score, corr, k):

    penalty = (
        2 * k *
        (score.max() + CFG["delta"])
    )

    bqm = BinaryQuadraticModel.empty(
        dimod.BINARY
    )

    # -----------------------------------
    # LINEAR TERMS
    # -----------------------------------

    for i in range(CFG["n_features"]):

        bqm.add_variable(
            i,
            -float(score[i])
        )

    # -----------------------------------
    # QUADRATIC TERMS
    # -----------------------------------

    for i in range(CFG["n_features"]):

        for j in range(i + 1,
                       CFG["n_features"]):

            r = corr[i, j]

            if r > CFG["corr_thresh"]:

                bqm.add_interaction(
                    i,
                    j,
                    CFG["delta"] * float(r)
                )

    # -----------------------------------
    # CARDINALITY CONSTRAINT
    # -----------------------------------

    bqm.update(
        dimod.generators.combinations(
            range(CFG["n_features"]),
            k,
            strength=penalty
        )
    )

    return bqm


# =========================================================
# SA SOLVER VIA QCLEF
# =========================================================

def solve_sa_qclef(bqm, k):

    sampler = SimulatedAnnealingSampler()

    sampleset = qa.submit(
        sampler,
        SimulatedAnnealingSampler.sample,
        bqm,
        label=f"1A_SA_k{k}",
        num_reads=CFG["sa_num_reads"],
        seed=CFG["random_state"]
    )

    best = sampleset.first.sample

    problem_id = sampleset.info.get(
        "problem_id",
        str(uuid.uuid4())
    )

    return best, problem_id


# =========================================================
# EVALUATION
# =========================================================

def evaluate_ndcg(selected):

    clf = LogisticRegression(
        max_iter=500
    )

    clf.fit(
        X_train[:, selected],
        y_train
    )

    preds = clf.predict_proba(
        X_dev[:, selected]
    )[:, 1]

    ndcg_vals = []

    offset = 0

    groups = compute_groups(q_dev)

    for g in groups:

        sl = slice(offset, offset + g)

        true_rel = y_dev[sl].reshape(1, -1)

        pred_rel = preds[sl].reshape(1, -1)

        if true_rel.sum() > 0:

            ndcg_vals.append(
                ndcg_score(
                    true_rel,
                    pred_rel,
                    k=10
                )
            )

        offset += g

    return float(np.mean(ndcg_vals))


# =========================================================
# SUBMISSION WRITER
# =========================================================

def save_submission(
    selected,
    problem_id,
    k,
    ndcg
):

    fname = (
        f"1A_SA_"
        f"{CFG['group_name']}_"
        f"{CFG['method_id']}_"
        f"k{k}.txt"
    )

    path = os.path.join(
        CFG["submit_dir"],
        fname
    )

    with open(path, "w") as f:

        for feat in selected:

            # 1-indexed feature IDs
            f.write(f"{feat + 1}\n")

        f.write(f"['{problem_id}']\n")

    print("\nSaved submission:")
    print(path)

    print("nDCG@10 =", round(ndcg, 4))


# =========================================================
# MAIN LOOP
# =========================================================

results = []

print("\nStarting QUBO sweep...")

for k in CFG["k_values"]:

    print("\n===================================")
    print("k =", k)
    print("===================================")

    # -----------------------------------
    # BUILD BQM
    # -----------------------------------

    bqm = build_bqm(
        score,
        corr,
        k
    )

    print(
        "Variables:",
        bqm.num_variables
    )

    print(
        "Interactions:",
        bqm.num_interactions
    )

    # -----------------------------------
    # SOLVE
    # -----------------------------------

    start = time.time()

    best, problem_id = solve_sa_qclef(
        bqm,
        k
    )

    elapsed = time.time() - start

    # -----------------------------------
    # SELECT FEATURES
    # -----------------------------------

    selected = [
        i
        for i, v in best.items()
        if v == 1
    ]

    selected = sorted(selected)

    print(
        "Selected features:",
        [x + 1 for x in selected]
    )

    print(
        "Solve time:",
        round(elapsed, 2),
        "seconds"
    )

    print(
        "Problem ID:",
        problem_id
    )

    # -----------------------------------
    # EVALUATE
    # -----------------------------------

    ndcg = evaluate_ndcg(
        selected
    )

    print(
        "nDCG@10:",
        round(ndcg, 4)
    )

    # -----------------------------------
    # SAVE SUBMISSION
    # -----------------------------------

    save_submission(
        selected,
        problem_id,
        k,
        ndcg
    )

    results.append({
        "k": k,
        "selected": selected,
        "ndcg": ndcg
    })


# =========================================================
# FINAL RESULTS
# =========================================================

print("\n===================================")
print("FINAL RESULTS")
print("===================================")

results = sorted(
    results,
    key=lambda x: -x["ndcg"]
)

for r in results:

    print(
        "k =",
        r["k"],
        "| nDCG@10 =",
        round(r["ndcg"], 4)
    )

best = results[0]

print("\nBEST RESULT")

print(
    "Best k:",
    best["k"]
)

print(
    "Best nDCG@10:",
    round(best["ndcg"], 4)
)

print(
    "Best features:",
    [x + 1 for x in best["selected"]]
)


Loading data...
Train shape: (41955, 46)
Dev shape: (13855, 46)

Computing Mutual Information...
Top MI features: [13  1 11  5 31]

Training Random Forest...
Top RF features: [23 37 21 39 16]

Building composite score...
Top Composite features: [ 1 11  5 23 21]

Computing feature redundancy...
Correlation matrix shape: (46, 46)

Starting QUBO sweep...

k = 8
Variables: 46
Interactions: 1035
Selected features: [1, 5, 11, 13, 17, 21, 38, 42]
Solve time: 4.11 seconds
Problem ID: SA-7152
nDCG@10: 0.4184

Saved submission:
/config/workspace/submissions/1A_SA_FAST-NUCES_MI_RF_SA_k8.txt
nDCG@10 = 0.4184

k = 10
Variables: 46
Interactions: 1035
Selected features: [15, 17, 18, 20, 23, 28, 32, 41, 42, 45]
Solve time: 4.37 seconds
Problem ID: SA-7153
nDCG@10: 0.4666

Saved submission:
/config/workspace/submissions/1A_SA_FAST-NUCES_MI_RF_SA_k10.txt
nDCG@10 = 0.4666

k = 12
Variables: 46
Interactions: 1035
Selected features: [5, 11, 12, 13, 19, 20, 23, 33, 38, 41, 43, 45]
Solve time: 4.4 seconds
P

In [5]:
"""
QuantumCLEF 2025 — Task 1A
Quantum Feature Selection using REAL Quantum Annealing (D-Wave QPU)

Uses:
- Mutual Information
- Stability across folds
- Correlation redundancy penalty
- REAL Quantum Annealer via QClef API
- nDCG@10 evaluation
- Submission writer

Dataset format:
train.csv
dev.csv

Expected columns:
relevance, query_id, 1, 2, ... 46
"""

import os
import uuid
import warnings
import numpy as np
import pandas as pd

import dimod

from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import ndcg_score
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression

from dwave.system import DWaveSampler, EmbeddingComposite
from dwave.preprocessing import ScaleComposite

from qclef import qa_access as qa

warnings.filterwarnings("ignore")


# =========================================================
# CONFIG
# =========================================================

CFG = {

    # -------------------------
    # DATA
    # -------------------------
    "train_path":
        "/config/workspace/datasets/Tasks/1A/MQ2007_train.csv",

    "dev_path":
        "/config/workspace/datasets/Tasks/1A/MQ2007_dev.csv",

    # -------------------------
    # OUTPUT
    # -------------------------
    "submission_dir":
        "/config/workspace/submissions",

    # -------------------------
    # FEATURE SELECTION
    # -------------------------
    "k": 12,

    # -------------------------
    # QUBO WEIGHTS
    # -------------------------
    "alpha": 0.6,     # MI
    "beta": 0.4,      # stability
    "delta": 0.3,     # redundancy penalty

    # -------------------------
    # REDUNDANCY
    # -------------------------
    "corr_threshold": 0.15,

    # -------------------------
    # QPU
    # -------------------------
    "num_reads": 1000,
    "chain_strength": 5.0,

    # -------------------------
    # SUBMISSION
    # -------------------------
    "group_name": "FAST-NUCES",
    "submission_id": "QPU_K12"
}

os.makedirs(CFG["submission_dir"], exist_ok=True)


# =========================================================
# LOAD CSV
# =========================================================

def load_csv(path):

    df = pd.read_csv(path)

    y = df["relevance"].values.astype(np.int32)

    query_ids = df["query_id"].values.astype(np.int32)

    feature_cols = [str(i) for i in range(1, 47)]

    X = df[feature_cols].values.astype(np.float64)

    return X, y, query_ids


print("\nLoading datasets...")

X_train, y_train, q_train = load_csv(CFG["train_path"])

X_dev, y_dev, q_dev = load_csv(CFG["dev_path"])


# =========================================================
# NORMALIZATION
# =========================================================

print("\nNormalizing features...")

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)

X_dev = scaler.transform(X_dev)

n_features = X_train.shape[1]

print("Train shape:", X_train.shape)
print("Dev shape:", X_dev.shape)


# =========================================================
# STEP 1 — MI + STABILITY
# =========================================================

print("\nComputing MI stability...")

gkf = GroupKFold(n_splits=5)

mi_scores = []

for fold, (tr_idx, va_idx) in enumerate(
    gkf.split(X_train, y_train, q_train)
):

    X_tr = X_train[tr_idx]
    y_tr = y_train[tr_idx]

    mi = mutual_info_regression(
        X_tr,
        y_tr,
        random_state=42
    )

    mi_scores.append(mi)

    print(f"Fold {fold+1} MI mean: {mi.mean():.4f}")


mi_scores = np.array(mi_scores)

mi_mean = mi_scores.mean(axis=0)

mi_std = mi_scores.std(axis=0)

stability = mi_mean / (mi_std + 1e-9)


# =========================================================
# NORMALIZE SCORES
# =========================================================

mi_norm = (
    (mi_mean - mi_mean.min()) /
    (mi_mean.max() - mi_mean.min() + 1e-9)
)

stab_norm = (
    (stability - stability.min()) /
    (stability.max() - stability.min() + 1e-9)
)


# =========================================================
# STEP 2 — COMPOSITE FEATURE SCORE
# =========================================================

print("\nBuilding feature scores...")

score = (
    CFG["alpha"] * mi_norm +
    CFG["beta"] * stab_norm
)

score = (
    (score - score.min()) /
    (score.max() - score.min() + 1e-9)
)


# =========================================================
# STEP 3 — CORRELATION REDUNDANCY
# =========================================================

print("\nComputing feature correlations...")

corr = np.abs(np.corrcoef(X_train.T))


# =========================================================
# STEP 4 — BUILD BQM
# =========================================================

print("\nBuilding Binary Quadratic Model...")

bqm = dimod.BinaryQuadraticModel.empty(dimod.BINARY)


# -------------------------
# DIAGONAL TERMS
# maximize feature quality
# -------------------------

for i in range(n_features):

    bqm.add_variable(i, -float(score[i]))


# -------------------------
# OFF-DIAGONAL TERMS
# redundancy penalty
# -------------------------

edge_count = 0

for i in range(n_features):

    for j in range(i + 1, n_features):

        r = corr[i, j]

        if r > CFG["corr_threshold"]:

            bqm.add_interaction(
                i,
                j,
                CFG["delta"] * r
            )

            edge_count += 1


print("Correlation edges:", edge_count)


# -------------------------
# EXACTLY k FEATURES
# -------------------------

penalty = 20

bqm.update(
    dimod.generators.combinations(
        n_features,
        CFG["k"],
        strength=penalty
    )
)


# =========================================================
# STEP 5 — QUANTUM ANNEALING (QPU)
# =========================================================

print("\nSubmitting to D-Wave Quantum Annealer...")

sampler = EmbeddingComposite(
    ScaleComposite(
        DWaveSampler()
    )
)

sampleset = qa.submit(
    sampler,
    EmbeddingComposite.sample,
    bqm,
    label="1A_QPU_k12",
    num_reads=CFG["num_reads"],
    chain_strength=CFG["chain_strength"]
)

best = sampleset.first.sample

problem_id = sampleset.info.get(
    "problem_id",
    str(uuid.uuid4())
)

print("\nQuantum submission successful")
print("Problem ID:", problem_id)


# =========================================================
# STEP 6 — SELECT FEATURES
# =========================================================

selected = sorted([
    i for i, v in best.items()
    if v == 1
])

print("\nSelected Features:")
print([i + 1 for i in selected])

print("Total selected:", len(selected))


# =========================================================
# STEP 7 — TRAIN RANKING MODEL
# =========================================================

print("\nTraining evaluation model...")

Xtr = X_train[:, selected]

Xdv = X_dev[:, selected]

clf = LogisticRegression(max_iter=1000)

clf.fit(Xtr, y_train)

pred = clf.predict_proba(Xdv)[:, 1]


# =========================================================
# STEP 8 — COMPUTE nDCG@10
# =========================================================

print("\nEvaluating nDCG@10...")

query_scores = []

unique_queries = np.unique(q_dev)

for q in unique_queries:

    mask = q_dev == q

    y_true = y_dev[mask]

    y_score = pred[mask]

    if np.sum(y_true) > 0:

        ndcg = ndcg_score(
            [y_true],
            [y_score],
            k=10
        )

        query_scores.append(ndcg)


mean_ndcg = np.mean(query_scores)

print("\nnDCG@10 =", round(mean_ndcg, 4))


# =========================================================
# STEP 9 — SAVE SUBMISSION
# =========================================================

submission_name = (
    f"1A_QPU_FAST-NUCES_k12.txt"
)

submission_path = os.path.join(
    CFG["submission_dir"],
    submission_name
)

with open(submission_path, "w") as f:

    # feature IDs must be 1-indexed
    for feat in selected:

        f.write(f"{feat + 1}\n")

    # last line = problem ID
    f.write(f"['{problem_id}']\n")


print("\nSubmission saved:")
print(submission_path)


Loading datasets...

Normalizing features...
Train shape: (41955, 46)
Dev shape: (13855, 46)

Computing MI stability...
Fold 1 MI mean: 0.0310
Fold 2 MI mean: 0.0304
Fold 3 MI mean: 0.0310
Fold 4 MI mean: 0.0307
Fold 5 MI mean: 0.0305

Building feature scores...

Computing feature correlations...

Building Binary Quadratic Model...
Correlation edges: 281

Submitting to D-Wave Quantum Annealer...

Quantum submission successful
Problem ID: d774f134-bb38-4f19-a34e-1d693b6deecf

Selected Features:
[1, 4, 5, 8, 9, 14, 16, 19, 20, 23, 24, 26, 33, 34, 40, 41, 44, 46]
Total selected: 18

Training evaluation model...

Evaluating nDCG@10...

nDCG@10 = 0.4871

Submission saved:
/config/workspace/submissions/1A_QPU_FAST-NUCES_k12.txt
